# Paramétrage de l'environnement de travail et import des packages

In [45]:
import sys
from pathlib import Path

In [44]:
ROOT = Path.cwd().parents[0]

RAW_DATA = ROOT / "01_data" / "01_raw"
PROCESSED_DATA = ROOT / "01_data" / "02_processed"
MODEL_DATA = ROOT / "04_model"

%load_ext autoreload
%autoreload 2
sys.path.append(str(ROOT / "03_fonctions"))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
import pandas as pd
import streamlit as st
import shap
import joblib
import json
from huggingface_hub import InferenceClient
from fonctions_perso.machine_learning import BinaryMetricsSimple, graphique_courbe_pr, graphique_courbe_calibration, graphique_courbe_roc, ThresholdCostOptimizer
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from sklearn import set_config

**Chargement des données/objets utiles**

À éxécuter une seule fois lors du démarrage

In [ ]:
@st.cache_resource
def load_models():

    modele_propre = joblib.load(MODEL_DATA / "fraud_detection_model_xgboost.joblib")
    modele_shap = joblib.load(MODEL_DATA / "fraud_detection_xgb_pas_calibre.joblib")
 
    return modele_propre, modele_shap


def build_features_for_model(X_raw: pd.DataFrame, modele_shap):
    
    set_config(transform_output="pandas")

    preprocess = modele_shap[:-1]
    X_processed = preprocess.transform(X_raw)

    ohe_names = modele_shap.named_steps["ohe"].get_feature_names_out()

    try:
        if_names = modele_shap.named_steps["isolation_forest"].get_feature_names_out()
    except AttributeError:
        extra = X_processed.shape[1] - len(ohe_names)
        if_names = [f"iforest_feature_{i}" for i in range(extra)]

    feature_names = list(ohe_names) + list(if_names)
    return pd.DataFrame(X_processed, columns=feature_names)



@st.cache_data
def load_sample_data():

    data_1_sample = joblib.load(PROCESSED_DATA / "1_fraud_data_sample.joblib")
    data_0_sample = joblib.load(PROCESSED_DATA / "0_fraud_data_sample.joblib")

    sample_data = pd.concat([data_1_sample,data_0_sample], axis = 0).drop(["target"], axis = 1).sample(10)

    return sample_data


def score_transaction(input_dict):
    """
    input_dict : dict avec les features brutes
    renvoie : proba, shap_values, X_shap
    """
    modele_propre, modele_shap = load_models()
    explainer = joblib.load(PROCESSED_DATA / "explainer.joblib")

    # 1. DataFrame brut
    X_raw = pd.DataFrame([input_dict])

    # 2. Features pour le modèle (même espace que pendant le training)
    X_shap = build_features_for_model(X_raw, modele_shap)

    # 3. Proba avec le modèle calibré
    proba = modele_propre.predict_proba(X_raw)[:, 1][0]

    # 4. SHAP sur les mêmes features
    shap_values = explainer.shap_values(X_shap)

    return proba, shap_values, X_shap



def plot_local_shap(shap_values, X):
    """
    Affiche un waterfall plot des contributions SHAP pour une ligne.
    """
    explainer = joblib.load(PROCESSED_DATA / "explainer.joblib")

    # shap_values: array (1, n_features) -> on prend la première ligne
    sv_row = shap_values[0, :]

    # On construit un Explanation pour utiliser shap.plots.waterfall
    shap_expl = shap.Explanation(
        values=sv_row,
        base_values=explainer.expected_value,
        data=X.iloc[0, :].values,
        feature_names=X.columns.tolist()
    )

    shap.plots.waterfall(shap_expl, max_display=15, show=False)
    st.pyplot(bbox_inches="tight", dpi=100)




def build_prompt_for_transaction(proba, input_dict, top_features):
    """
    proba: float (probabilité de fraude)
    input_dict: dict des features brutes de la transaction
    top_features: liste de tuples (feature_name, shap_value) triés par importance absolue
    """
    decision = "bloquée (suspecte)" if proba >= 0.226 else "acceptée"
    top_str = "\n".join(
        [f"- {name} (contribution SHAP = {value:+.3f})" for name, value in top_features]
    )

    prompt = f"""
Tu es un analyste fraude dans une banque française.
Explique en français, de façon claire et compréhensible pour un client,
pourquoi la transaction suivante est {decision} par le modèle de détection de fraude.

Contexte :
- Probabilité estimée de fraude : {proba:.3f}
- Décision automatique du modèle : {decision}

Caractéristiques de la transaction :
- Montant : {input_dict.get("montant_transaction")}
- Type de magasin : {input_dict.get("type_magasin")}
- Heure de la transaction : {input_dict.get("heure_transaction")}h
- État du client : {input_dict.get("etat_client")}
- Âge du client : {input_dict.get("age_client")}
- Distance domicile–magasin : {input_dict.get("distance_domicile_magasin")}

Principales variables ayant influencé le modèle (SHAP) :
{top_str}

Consignes :
- Adopte un ton professionnel mais pédagogique.
- Ne parle pas de “SHAP” ni de “modèle XGBoost”, parle de “algorithme interne de détection de fraude”.
- Mets en avant les éléments qui augmentent le risque et ceux qui le réduisent.
- Rédige 1 à 2 paragraphes maximum.
"""
    return prompt


hugging_face_json = "hugging_face_token.json"
with open(ROOT / hugging_face_json, "r") as f:
    secrets = json.load(f)

TOKEN = secrets["token_hugging_face"]

client = InferenceClient(
    model="mistralai/Mistral-7B-Instruct-v0.2",  
    token=TOKEN
)


def call_llm_explanation(proba, shap_values, input_dict, feature_names, k_top=5):
    # shap_values: array 1D des contributions pour chaque feature (transaction unique)
    # feature_names: liste des noms de features alignés avec shap_values
    import numpy as np

    # top k features par importance absolue
    idx_sorted = np.argsort(-np.abs(shap_values))
    top_idx = idx_sorted[:k_top]
    top_features = [(feature_names[i], float(shap_values[i])) for i in top_idx]

    prompt = build_prompt_for_transaction(proba, input_dict, top_features)

    # appel Hugging Face
    output = client.text_generation(
        prompt,
        max_new_tokens=250,
        temperature=0.4,
        top_p=0.9,
        do_sample=True,
    )[0]["generated_text"]  # format de retour

    return output


2026-03-14 15:39:02.766 No runtime found, using MemoryCacheStorageManager


**Onglet 1**

In [ ]:
def tab_ml_metrics():
    st.header("Métriques du modèle")

    y_train = joblib.load(PROCESSED_DATA / "y_train.joblib")
    train_proba = joblib.load(PROCESSED_DATA / "train_proba.joblib")
    y_test = joblib.load(PROCESSED_DATA / "y_test_for_shap.joblib")
    test_proba = joblib.load(PROCESSED_DATA / "test_proba.joblib")
    seuil_decision = 0.226

    y_pred_test = (test_proba >= seuil_decision).astype(int)

    metrics_obj = BinaryMetricsSimple(
        y_true=y_test,
        y_pred=y_pred_test,
        y_proba=test_proba
    )

    st.subheader("Métriques globales (test)")

    col_left, col_right = st.columns([1.4, 1], gap="medium")

    with col_left:
        df_metrics = metrics_obj.get_metrics_df()
        st.dataframe(df_metrics, use_container_width=False)

    with col_right:
        st.text("Matrice de confusion")
        st.text(metrics_obj.print_confusion_matrix())

    st.markdown("---")

    col_pr, col_cal = st.columns([1, 1], gap="medium")

    with col_pr:
        st.subheader("Précision–Rappel")
        fig_pr, ax_pr = plt.subplots(figsize=(5, 4))
        graphique_courbe_pr(
            y_train=y_train,
            train_proba=train_proba,
            y_test=y_test,
            test_proba=test_proba,
            figsize=(5, 4),
            save_path=None,
            ax=ax_pr if "ax" in graphique_courbe_pr.__code__.co_varnames else None,
        )
        st.pyplot(fig_pr, width="content")  

    with col_cal:
        st.subheader("Calibration")
        fig_cal, ax_cal = plt.subplots(figsize=(5, 4))
        graphique_courbe_calibration(
            y_train=y_train,
            train_proba=train_proba,
            y_test=y_test,
            test_proba=test_proba,
            n_bins=10,
            figsize=(5, 4),
            save_path=None,
            ax=ax_cal if "ax" in graphique_courbe_calibration.__code__.co_varnames else None,
        )
        st.pyplot(fig_cal, width="content")  

**Onglet 2**

In [ ]:
def tab_business_metrics():
    st.header("Métriques métier & coût du seuil")

    y_validation = joblib.load(PROCESSED_DATA / "y_validation.joblib")
    proba_validation = joblib.load(PROCESSED_DATA / "proba_validation.joblib")

    # 1. Entrée des coûts métier
    col1, col2 = st.columns(2)
    with col1:
        cost_fp = st.number_input(
            "Coût d’un faux positif (transaction du client bloquée à tort)",
            value=25,
            min_value=0
        )
    with col2:
        cost_fn = st.number_input(
            "Coût d’une fraude non détectée",
            value=125,
            min_value=0
        )

    # 2. Fit optimiseur sur l’échantillon de validation
    optimizer = ThresholdCostOptimizer(cost_fp=cost_fp, cost_fn=cost_fn, n_thresholds=200)
    optimizer.fit(y_validation, proba_validation)

    seuils = optimizer.seuils_
    costs = optimizer.costs_
    best_t = optimizer.get_best_threshold()
    best_cost = optimizer.get_best_cost()

    st.subheader("Coût total en fonction du seuil de décision")

    # 3. Graphique Plotly interactif (UNE seule courbe)
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=seuils,
            y=costs,
            mode="lines",
            name="Coût total",
            line=dict(color="#1f77b4"),
            hovertemplate=(
                "Seuil: %{x:.4f}<br>"  # 4 décimales pour le seuil
                "Coût total: %{y:,.0f} €<extra></extra>"  # séparateur milliers + €
            ),
        )
    )

    # Point seuil optimal
    fig.add_trace(
        go.Scatter(
            x=[best_t],
            y=[best_cost],
            mode="markers",
            name=f"Seuil optimal ({best_t:.4f})",
            marker=dict(color="red", size=9),
            hovertemplate=(
                "Seuil optimal: %{x:.4f}<br>"
                "Coût minimum: %{y:,.0f} €<extra></extra>"
            ),
        )
    )

    fig.update_layout(
        xaxis_title="Seuil de décision",
        yaxis_title="Coût total (FP/FN) en €",
        title="Coût total en fonction du seuil de décision (échantillon de validation)",
        hovermode="x unified",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.2,
            xanchor="center",
            x=0.5,
        ),
        margin=dict(l=40, r=20, t=60, b=60),
    )

    # nouvelle API de largeur : width="stretch" au lieu de use_container_width
    st.plotly_chart(fig, width="stretch")

    # 4. Résumé texte métier
    st.markdown(
        f"""
        - **Seuil optimal (validation)** : {best_t:.4f}, coût minimum ≈ {best_cost:,.0f} €  
        - La courbe permet de voir comment le seuil modifie le **coût total**, en combinant
          faux positifs (clients gênés inutilement) et fraudes non détectées.
        """
    )


**Onglet 3**

In [50]:
def tab_transactions_and_explanations():
    st.header("Transactions et explications locales")

    sample_data = load_sample_data()

    st.markdown("Sélectionnez une transaction pour voir l’explication détaillée.")
    st.dataframe(sample_data)

    idx = st.number_input(
        "Indice de la transaction à expliquer",
        min_value=0,
        max_value=len(sample_data) - 1,
        value=0,
        step=1
    )

    row = sample_data.iloc[idx]
    st.write("Transaction sélectionnée :", row.to_dict())

    # scoring + SHAP local
    proba, shap_values, X_shap = score_transaction(row.to_dict())
    st.write(f"Probabilité estimée de fraude : {proba:.3f}")

    st.subheader("Explication locale (SHAP)")
    plot_local_shap(shap_values, X_shap)

    if st.button("Générer une explication métier (LLM)"):
        # shap_values pour une seule ligne -> vecteur 1D
        shap_1d = shap_values[0, :]
        feature_names = X_shap.columns.tolist()

        explanation = call_llm_explanation(
            proba=proba,
            shap_values=shap_1d,
            input_dict=row.to_dict(),
            feature_names=feature_names,
            k_top=5
        )
        st.markdown(explanation)


**Onglet 4**

In [51]:
def tab_simulator():
    st.header("Votre transaction serait-elle considérée comme frauduleuse ?")

    # quelques inputs utilisateur
    montant = st.number_input("Montant de la transaction", min_value=0.0, value=50.0)
    type_magasin = st.selectbox("Type de magasin", ["grocery_pos", "gas_transport", "shopping_pos", "misc_net"])
    heure = st.selectbox("Heure de la transaction", list(range(0, 24)))
    etat = st.selectbox("État du client", ["CA", "NY", "TX", "autre"])
    age = st.slider("Âge du client", 18, 90, 40)
    distance = st.number_input("Distance domicile–magasin (km)", min_value=0.0, value=5.0)

    if st.button("Évaluer la transaction"):
        input_dict = {
            "montant_transaction": montant,
            "type_magasin": type_magasin,
            "heure_transaction": heure,
            "etat_client": etat,
            "age_client": age,
            "distance_domicile_magasin": distance,
        }
        proba, shap_values, X = score_transaction(input_dict)
        st.write(f"Probabilité estimée de fraude : {proba:.3f}")

        decision = "ALERTE FRAUDE" if proba >= 0.226 else "ACCEPTÉE"
        st.write(f"Décision (seuil 0,226) : **{decision}**")

        st.subheader("Explication locale (SHAP)")
        plot_local_shap(shap_values, X)


**Main Streamlit**

In [52]:
def main():
    st.set_page_config(page_title="Détection de fraude carte bancaire", layout="wide")

    st.title("Fraude carte bancaire – Modèle XGBoost explicable")

    tab1, tab2, tab3, tab4 = st.tabs([
        "Métriques ML",
        "Métriques métier & coûts",
        "Transactions & explications",
        "Simulateur de transaction",
    ])

    with tab1:
        tab_ml_metrics()
    with tab2:
        tab_business_metrics()
    with tab3:
        tab_transactions_and_explanations()
    #with tab4:
    #    tab_simulator()


if __name__ == "__main__":
    main()


2026-03-14 15:39:21.026 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:21.027 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:21.028 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:21.030 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:21.033 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:21.034 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:21.036 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:21.040 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

MATRICE DE CONFUSION
                      Prédit Négatif | Prédit Positif
Réel Négatif              279845 |            740
Réel Positif                 199 |            737


c:\Users\user\Desktop\data_science_documents\data_science_projets_perso\projet_fraud_detection\fraud-detection\03_fonctions\fonctions_perso\machine_learning.py:171: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown

2026-03-14 15:39:22.621 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.



RÉSULTATS PRÉCISION-RAPPEL
AP Train     : 0.7868
AP Test      : 0.7209
Écart        : 0.0659
Baseline     : 0.0033
⚠️  Attention : écart élevé (possible overfitting)
Lift vs baseline : 216.83x


2026-03-14 15:39:23.015 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.018 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.022 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.035 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.038 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.040 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
c:\Users\user\Desktop\data_science_documents\data_science_projets_perso\projet_fraud_detection\fraud-detection\03_fonctions\fonctions_perso\machine_learning.py:234: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown

202


RÉSULTATS DE CALIBRATION
Brier Score Train : 0.0022
Brier Score Test  : 0.0016
Écart Brier       : 0.0006

ECE Train         : 0.0308
ECE Test          : 0.0775

---------------------------------------------
INTERPRÉTATION :
✓ Excellente calibration (Brier < 0.1)
⚠️  Erreur de calibration modérée (ECE < 0.1)


2026-03-14 15:39:23.716 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.717 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.718 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.726 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.727 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.733 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.791 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-14 15:39:23.792 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [37]:
data_1_sample = joblib.load(PROCESSED_DATA / "1_fraud_data_sample.joblib")
data_0_sample = joblib.load(PROCESSED_DATA / "0_fraud_data_sample.joblib")

sample_data = pd.concat([data_1_sample,data_0_sample], axis = 0).drop(["target"], axis = 1).sample(10)

In [39]:
row = sample_data.iloc[4]

In [40]:
row.to_dict()

{'type_magasin': 'personal_care',
 'montant_transaction': 51.43,
 'etat_client': 'OR',
 'population_ville_client': 1288,
 'heure_transaction': '22',
 'distance_domicile_magasin': 34.24791483217234,
 'age_client': 84}